# CurvLearn — Design A: is *flat* an attractor?

Sweep the **initial curvature** κ₀ against the **embedding scale** and watch whether a freely
learnable κ actually moves. Prediction (κ-stereographic expansion): a frozen basin around κ₀=0
that narrows as the embedding scale grows — because ∂L/∂κ vanishes as r² – r⁴ near flat.

Runtime → change to a **GPU** (A100 ideal): *Runtime ▸ Change runtime type ▸ GPU*.

**Persistence:** results and figures are written to **Google Drive**, and the sweep
checkpoints after every cell — so a Colab disconnect costs at most one cell, and re-running
resumes where it stopped.


## 1. GPU check


In [ ]:
!nvidia-smi -L || echo 'no GPU — will run on CPU (slower but fine)'

## 2. Get the code and install the package
`pip install -e .` installs `curvlearn` (and its deps) into the environment, so `import
curvlearn` resolves everywhere — including in the `!python tests/...` subprocess below.


In [ ]:
import os
if not os.path.isdir('CurvLearn') and os.path.basename(os.getcwd()) != 'CurvLearn':
    !git clone https://github.com/TheFausap/CurvLearn.git
if os.path.basename(os.getcwd()) != 'CurvLearn':
    os.chdir('CurvLearn')
!pip -q install -e .
print('cwd:', os.getcwd())

## 3. Mount Google Drive and pick a persistent output folder
Everything below writes to `OUTDIR`. On Colab that is a folder in your Drive, so results and
figures survive the runtime being recycled. Off Colab it falls back to a local `results/`.


In [ ]:
import os, torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTDIR = '/content/drive/MyDrive/CurvLearn/results'
except ModuleNotFoundError:
    OUTDIR = 'results'          # not on Colab: local folder
os.makedirs(OUTDIR, exist_ok=True)
print('device:', dev, '| results ->', OUTDIR)

## 4. Sanity: geometry matches the dependency-free reference
Confirms the tensor ops equal the pure-Python formulas, the κ→0 limit, and the (8/3)·r⁴
curvature-gradient law that the whole experiment rests on.


In [ ]:
!python tests/test_geometry.py

## 5. One quick run (smoke test, ~seconds)
A single non-flat model, 200 steps, just to confirm the loop and the κ logging work.


In [ ]:
from curvlearn.model import ModelConfig
from curvlearn.train import TrainConfig, run_training
from curvlearn.data import make_dataset

data = make_dataset('dyck', k=3, max_depth=12)
res = run_training(ModelConfig(vocab_size=data.vocab_size, kappa_init=-1.0, emb_scale=1.0),
                   data, TrainConfig(steps=200, device=dev, log_every=25))
print(f"kappa {res['kappa_init']:+.2f} -> {res['kappa_final']:+.3f} | loss {res['final_loss']:.3f}")

## 6. The Design A sweep (checkpointed to Drive)
27 cells (9 κ₀ × 3 embedding scales), 2000 steps each. Minutes on an A100.
Results are saved to `OUTDIR/designA_results.json` after every cell; if the runtime drops,
just re-run this cell and it resumes. Drop `steps` for a quick pass, or edit
`experiments/designA.yaml` for larger grids.


In [ ]:
import json
from curvlearn.sweep import run_sweep, default_spec

spec = default_spec()
spec['train']['device'] = dev
spec['train']['steps'] = 2000          # <- lower for a quick pass
ckpt = os.path.join(OUTDIR, 'designA_results.json')
out = run_sweep(spec, checkpoint_path=ckpt, resume=True)   # incremental + resumable to Drive
print('cells:', len(out['cells']), '| wall (s):', round(out['wall_seconds']), '| saved ->', ckpt)

## 7. The phase diagram + curvature trajectories (saved to Drive)


In [ ]:
from curvlearn import plot
from IPython.display import Image, display
pd_png = os.path.join(OUTDIR, 'designA_phase_diagram.png')
tj_png = os.path.join(OUTDIR, 'designA_kappa_trajectories.png')
plot.phase_diagram(out, pd_png)
plot.trajectories(out, tj_png)
display(Image(pd_png)); display(Image(tj_png))
print('figures saved to', OUTDIR)

## 8. Read the result
- **Frozen basin around κ₀=0?** small |Δκ| in the middle columns = flat is an attractor.
- **Basin narrows as embedding scale grows?** the two-way dependence is the real signature.
- **Escaped κ settles at κ<0 on Dyck?** hierarchy prefers hyperbolic.

If all three hold, the premise for Designs B–D (scheduled / non-flat curvature) is confirmed:
curvature must be *placed and held* off flat, not left to a gradient that is pulled toward zero.
Next: `kappa_mode='schedule'` in `TrainConfig` is the hook for Design B (curvature annealing).
